In [135]:
import numpy as np
from numpy import sqrt
import pandas as pd
import matplotlib.pyplot as plt
import math

In [136]:
class Measurement:
    def __init__(self, data: list[float] | np.ndarray, name: str | None = ""):
        data = data.astype(float) if isinstance(data, np.ndarray) else np.array(data, dtype=float)
        if data.ndim == 1:
            self.data = data
        else:
            raise ValueError(f"Measurements must be of shape (n,). Got {data.shape}.")
        self.name = name
        self.error = None

    def __repr__(self):
        if self.error is not None:
            return (
                f"Measurement({self.name}, data={self.data.tolist()}, error={self.error.tolist()})"
            )
        else:
            return f"Measurement({self.name}, {self.data.tolist()})"

    def setError(self, error: float | list[float] | np.ndarray):
        error = error if isinstance(error, np.ndarray) else np.array(error)
        if error.shape == tuple():
            self.error = np.full_like(self.data, error)
        elif error.shape == self.data.shape:
            self.error = error
        else:
            raise ValueError(
                f"Error must be scalar or of same shape as data. Expected () or {self.data.shape}, got {error.shape}."
            )
        return self

    def mean(self):
        if self.error is None:
            out_error = self.sem()
        else:
            syst_error = 1 / sqrt(np.sum(1 / self.error**2))
            out_error = sqrt(self.sem() ** 2 + syst_error**2)
        return Measurement([self.data.mean()], name=f"μ_{self.name}").setError(out_error)

    def std(self):
        # Da der Mittelwert aus den N Werten folgt, sind nur N - 1 Differenzen unabhängig.
        return self.data.std(ddof=1)

    def sem(self):
        return self.std() / sqrt(len(self.data))

In [137]:
m = Measurement([1,5], name="m").setError(0.1)
m.mean()

Measurement(μ_m, data=[3.0], error=[2.00124960961895])